# 06 — Create Addresses

For every ACTIVE subscription with a successfully resolved address (from
`05_Fetch_Subscriptions.ipynb`'s Voyager lookup), `PUT`s that address onto
its target account:

```
{{host}}/rest/SubscriberService/v1/subscribers/{accountcode}
```

`addLine1` / `addLine2` / `city` / `zip` / `state` (region ISO) all come
from the Voyager `ParsedAddress_*` columns — real values, not placeholders.

Returns each new address's `id`, saved as `ship_add_id` — this is what
`07_Create_Subscription_Orders.ipynb` uses as `shipAddId` on the order.

Subscriptions whose Voyager lookup didn't resolve are skipped here (flagged
with status `"skipped"`) — they need a manual address before their order can
be created.

Inactive subscriptions don't go through this notebook at all — see
`06_Attach_Inactive_Addresses.ipynb`, which runs AFTER this one finishes
(it needs every active address here to be created first, so it can find
each account's final default service address).


## 1. Setup

In [7]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))

from onebill_common import *  # noqa: F401,F403
from concurrent.futures import ThreadPoolExecutor, as_completed

logger = get_logger("create_addresses")

df_subscriptions = load_subscriptions_resolved()
logger.info(f"Loaded {len(df_subscriptions):,} subscriptions from 05_Fetch_Subscriptions.ipynb")


2026-07-27 10:33:45,905 [INFO] Loaded 264 subscriptions from 05_Fetch_Subscriptions.ipynb


## 2. Per-subscription address creation

In [8]:
def create_address_for_subscription(session: requests.Session, row: dict) -> dict:
    subscription_id = row["SubscriptionUSN"]
    account_number  = row["TargetAccountNumber"]

    result = {
        "SubscriptionUSN":      subscription_id,
        "TargetAccountNumber":  account_number,
        "addLine1":             row.get("ParsedAddress_addLine1"),
        "status":               "failed",
        "ship_add_id":          None,
        "error":                None,
    }

    if not row.get("ParsedAddress_parsed_ok"):
        result["status"] = "skipped"
        result["error"] = row.get("ParsedAddress_error") or "Voyager address lookup failed — needs manual address"
        return result

    status, ship_add_id, error = add_address_to_account(
        session,
        account_number,
        row["ParsedAddress_addLine1"],
        location_id=row.get("ParsedAddress_location_id") or str(subscription_id),
        address2=row.get("ParsedAddress_addLine2"),
        city=row.get("ParsedAddress_city") or "Christchurch",
        zip_code=row.get("ParsedAddress_postcode") or "1234",
        region_iso=row.get("ParsedAddress_region_iso"),
    )
    result["status"] = status
    result["ship_add_id"] = ship_add_id
    result["error"] = error

    if status == "created":
        logger.info(f"[OK] subscription {subscription_id} -> {account_number} address id={ship_add_id}")
    else:
        logger.error(f"[FAIL] subscription {subscription_id} — {error}")

    return result


## 3. Run (parallel driver)

In [9]:
def create_all_addresses(df: pd.DataFrame, max_workers: int = MAX_WORKERS) -> pd.DataFrame:
    session = new_session(max_workers=max_workers)
    rows = df.to_dict("records")
    total = len(rows)
    results = []
    logger.info(f"Creating addresses for {total:,} subscriptions with {max_workers} workers...")
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(create_address_for_subscription, session, row): row["SubscriptionUSN"] for row in rows}
        for i, future in enumerate(as_completed(futures), start=1):
            results.append(future.result())
            if i % 50 == 0 or i == total:
                ok = sum(1 for r in results if r["status"] == "created")
                logger.info(f"Progress: {i}/{total} — {ok} created so far")
    return pd.DataFrame(results)


df_address_results = create_all_addresses(df_subscriptions)
df_address_results.head(20)


2026-07-27 10:33:46,045 [INFO] Creating addresses for 264 subscriptions with 10 workers...
2026-07-27 10:33:54,942 [INFO] [OK] subscription V113061451 -> 99965692_fullbatch5 address id=141992
2026-07-27 10:33:56,920 [INFO] [OK] subscription V113062384 -> 99965692_fullbatch5 address id=141898
2026-07-27 10:33:59,477 [INFO] [OK] subscription V113062392 -> 99965692_fullbatch5 address id=141899
2026-07-27 10:34:01,102 [INFO] [OK] subscription V113062996 -> 99965692_fullbatch5 address id=141993
2026-07-27 10:34:07,698 [INFO] [OK] subscription V113063002 -> 99965692_fullbatch5 address id=141900
2026-07-27 10:34:14,333 [INFO] [OK] subscription V113063010 -> 99965692_fullbatch5 address id=141994
2026-07-27 10:34:20,281 [INFO] [OK] subscription V113063036 -> 99965692_fullbatch5 address id=141995
2026-07-27 10:34:26,458 [INFO] [OK] subscription V113063044 -> 99965692_fullbatch5 address id=141901
2026-07-27 10:34:32,340 [INFO] [OK] subscription V113063051 -> 99965692_fullbatch5 address id=141902


,SubscriptionUSN,TargetAccountNumber,addLine1,status,ship_add_id,error
0,V113062343,99965692_fullbatch5,NaN,skipped,None,no SupplierServiceID on this subscription
1,V113062616,99965692_fullbatch5,NaN,skipped,None,circuits lookup failed: HTTPSConnectionPool(ho...
2,V113062400,99965692_fullbatch5,NaN,skipped,None,circuits lookup failed: HTTPSConnectionPool(ho...
3,V113062962,99965692_fullbatch5,NaN,skipped,None,circuits lookup failed: HTTPSConnectionPool(ho...
4,V113062335,99965692_fullbatch5,NaN,skipped,None,no SupplierServiceID on this subscription
5,V113062418,99965692_fullbatch5,NaN,skipped,None,circuits lookup failed: HTTPSConnectionPool(ho...
6,V113062988,99965692_fullbatch5,NaN,skipped,None,circuits lookup failed: HTTPSConnectionPool(ho...
7,V113062970,99965692_fullbatch5,NaN,skipped,None,circuits lookup failed: HTTPSConnectionPool(ho...
8,V113062632,99965692_fullbatch5,NaN,skipped,None,circuits lookup failed: HTTPSConnectionPool(ho...
9,V113061451,99965692_fullbatch5,39 Chester Street West,created,141992,None


## 4. Failures / skips

In [10]:
not_created = df_address_results[df_address_results["status"] != "created"]
print(f"{len(not_created):,} / {len(df_address_results):,} addresses not created (failed or skipped)")
not_created.groupby("status").size()


17 / 264 addresses not created (failed or skipped)


status
skipped    17
dtype: int64

## 5. Save

In [11]:
save_df("address_results", df_address_results)


Saved 264 rows -> migration_data\06_address_creation_results.csv


## 6. Debug — raw single-row PUT with full request/response detail

For chasing a specific failure (e.g. a `401`/`405` with an empty reason
phrase). Prints exactly what's being sent and the FULL response body —
`raise_for_status()` alone only gives you the status line, not the body,
which is where OneBill actually explains itself.

Set `DEBUG_SUBSCRIPTION_USN` below to the failing subscription's USN.
Also runs a GET on the same account first (sanity check that the account
exists and the token/proxy header are even accepted), and compares against
one of the two Williams bucket accounts to see if the problem is
account-specific or affects every account (i.e. an auth/proxy problem).

In [12]:
DEBUG_SUBSCRIPTION_USN = "V113062392"  # <-- change this to whichever subscription is failing

debug_row = df_subscriptions.loc[df_subscriptions["SubscriptionUSN"] == DEBUG_SUBSCRIPTION_USN].iloc[0].to_dict()
account_number = debug_row["TargetAccountNumber"]

print("=== Basics ===")
print("SubscriptionUSN:      ", debug_row["SubscriptionUSN"])
print("TargetAccountNumber:  ", repr(account_number), "| type:", type(account_number).__name__)
print("CREATION_PROXY_ACCOUNT_NUMBER (.env):", repr(ONEBILL_PROXY_ACCT))
print("ONEBILL_BASE_URL:     ", ONEBILL_BASE_URL)

print("\n=== Token ===")
token = token_manager.get_token()
print("token (first 20 chars):", token[:20], "...")
print("token length:", len(token))
print("token_manager expires_at:", token_manager._expires_at, "| now:", datetime.now())

debug_session = new_session(max_workers=1)
sent_headers = {**debug_session.headers, **auth_headers()}
print("\n=== Headers that will actually be sent ===")
for k, v in sent_headers.items():
    print(f"  {k}: {'Bearer ***' + v[-8:] if k == 'Authorization' else v}")

url = f"{ONEBILL_SUBSCRIBER_URL}/{account_number}"
print("\n=== URL ===")
print(url)

print("\n=== Step 1: GET the account first (does it exist? is the token/proxy header even accepted?) ===")
get_resp = debug_session.get(url, headers=auth_headers(), timeout=30)
print("GET status_code:", get_resp.status_code)
print("GET response body (first 1500 chars):\n", get_resp.text[:1500])

print("\n=== Step 2: does the SAME PUT work against a known-good bucket account? ===")
print("(isolates: is this account-specific, or does every account fail — i.e. an auth/proxy problem?)")
for key, acct in TARGET_ACCOUNTS.items():
    bucket_acct_number = real_account_numbers.get(key, acct["account_number"]) if 'real_account_numbers' in dir() else acct["account_number"]
    bucket_get = debug_session.get(f"{ONEBILL_SUBSCRIBER_URL}/{bucket_acct_number}", headers=auth_headers(), timeout=30)
    print(f"  GET {key} ({bucket_acct_number}): {bucket_get.status_code}")

print("\n=== Step 3: the actual PUT, with full response body on failure ===")
payload = {
    "address": [{
        "zip":             clean(debug_row.get("ParsedAddress_postcode")) or "1234",
        "country":         "NEW ZEALAND",
        "state":           clean(debug_row.get("ParsedAddress_region_iso")),
        "county":          "",
        "defaultBilling":  "false",
        "city":            clean(debug_row.get("ParsedAddress_city")) or "Christchurch",
        "addLine1":        debug_row.get("ParsedAddress_addLine1"),
        "addLine2":        clean(debug_row.get("ParsedAddress_addLine2")) or "",
        "defaultShipping": "false",
        "addressAttribute": [{"key": "Location Id", "value": str(debug_row["SubscriptionUSN"])}],
    }]
}
print("Payload:\n", json.dumps(payload, indent=2))

put_resp = debug_session.put(url, headers=auth_headers(), json=payload, timeout=30)
print("\nPUT status_code:", put_resp.status_code)
print("PUT response headers:", dict(put_resp.headers))
print("PUT response body:\n", put_resp.text)


=== Basics ===
SubscriptionUSN:       V113062392
TargetAccountNumber:   '99965692_fullbatch5' | type: str
CREATION_PROXY_ACCOUNT_NUMBER (.env): 'import01'
ONEBILL_BASE_URL:      https://sandbox-sg.onebillsoftware.com

=== Token ===
token (first 20 chars): 3f65b182-509c-47e2-8 ...
token length: 36
token_manager expires_at: 2026-07-27 11:32:08.670835 | now: 2026-07-27 11:00:39.962871

=== Headers that will actually be sent ===
  User-Agent: python-requests/2.32.5
  Accept-Encoding: gzip, deflate
  Accept: */*
  Connection: keep-alive
  proxy_accountNumber: import01
  Content-Type: application/json
  Authorization: Bearer ***055046c4

=== URL ===
https://sandbox-sg.onebillsoftware.com/rest/SubscriberService/v1/subscribers/99965692_fullbatch5

=== Step 1: GET the account first (does it exist? is the token/proxy header even accepted?) ===
GET status_code: 200
GET response body (first 1500 chars):
 {"accountName":"Williams Internet Limited","accountNumber":"99965692_fullbatch5","accountId":"